# Ch 10　Streaming 與事件流

> **本 notebook 的範例不需要 API key**（`messages` 逐 token 那種才需要 LLM，這裡用 `updates`/`values`/`custom` 示範）。一律使用 **`version="v2"`** 的統一格式。

In [ ]:
!uv pip install -q langgraph

## 10.3–10.4　v2 統一格式：updates vs values

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    topic: str
    joke: str

def refine_topic(state): return {"topic": state["topic"] + " 加上貓咪"}
def generate_joke(state): return {"joke": f"為什麼{state['topic']}要上學？為了拿到聖代學位！🍨"}

graph = (StateGraph(State)
         .add_node(refine_topic).add_node(generate_joke)
         .add_edge(START, "refine_topic").add_edge("refine_topic", "generate_joke")
         .add_edge("generate_joke", END).compile())

print("=== updates：只給『這一步改了什麼』（日誌式）===")
for part in graph.stream({"topic": "冰淇淋"}, stream_mode="updates", version="v2"):
    # v2 下每個 chunk 都是 {type, ns, data}，形狀固定、不必猜
    print(f"[{part['type']}] {part['data']}")

print("\n=== values：每步後的『完整 state 快照』===")
for part in graph.stream({"topic": "冰淇淋"}, stream_mode="values", version="v2"):
    print(f"[{part['type']}] {part['data']}")

## 10.6　custom：用 get_stream_writer 吐自訂進度

當你想告訴使用者「現在在忙什麼」，而那既不是 state 也不是 token。

In [ ]:
from langgraph.config import get_stream_writer

class S(TypedDict):
    topic: str
    joke: str

def joke_with_progress(state: S):
    writer = get_stream_writer()
    writer({"status": "正在絞盡腦汁想笑話…"})    # 這會以 custom mode 串流出去
    return {"joke": f"為什麼{state['topic']}不會迷路？因為它總是跟著路標走。"}

g = (StateGraph(S).add_node(joke_with_progress)
     .add_edge(START, "joke_with_progress").add_edge("joke_with_progress", END).compile())

# 同時要 updates 與 custom：v2 下用 part["type"] 分流，乾淨俐落
for part in g.stream({"topic": "冰淇淋"}, stream_mode=["updates", "custom"], version="v2"):
    if part["type"] == "custom":
        print("⏳ 狀態：", part["data"]["status"])
    elif part["type"] == "updates":
        print("✅ 更新：", part["data"])

## 10.5　messages（需要 API key）：打字機效果

逐 token 串流。下面這段需要設好金鑰、且圖裡有會呼叫 LLM 的節點才會動——這裡只示意寫法。

In [ ]:
# for part in graph.stream(inputs, stream_mode="messages", version="v2"):
#     msg, metadata = part["data"]
#     print(msg.content, end="", flush=True)   # end="" + flush=True = 打字機
print("messages mode 寫法示意（需 API key 與 LLM 節點才會逐字浮現）。")

## 重點回顧

新程式一律 `version="v2"`：每個 chunk 都是 `{type, ns, data}`。`updates` 看走到哪步、`values` 看完整全貌、`messages` 打字機、`custom`（用 `get_stream_writer`）自訂進度。